# To Do: Package the classification workflow

Using the same pattern from this lab (data.py -> features.py -> model.py ->
validate.py -> __init__.py), turn the classification code from Lab 3
(Breast Cancer dataset) into its own package. This time, go a step further
than just training one fixed model.

T1 --- model.py should not train just one model. Use cross-validation to
compare LogisticRegression, DecisionTreeClassifier, and
RandomForestClassifier, and automatically save whichever one scores best
(instead of hardcoding the winner yourself).

T2 --- validate.py should check that at least 3 of the input measurements
fall within a realistic range (e.g. radius_mean and area_mean can't be
negative or absurdly large) --- not just "is this a number".

T3 --- predict.py should print both the prediction (malignant/benign) AND
the model's confidence for that prediction (use predict_proba).

T4 --- add a metrics.py module with one function that prints a
confusion matrix and classification report for the saved model, so anyone
can check its performance without retraining.

In [1]:
from pathlib import Path

Path("breast_cancer").mkdir(exist_ok=True)

In [2]:
%%writefile breast_cancer/data.py
from pathlib import Path
import pandas as pd


def load_data():
    path = Path("data/breast-cancer.csv")
    return pd.read_csv(path)

Writing breast_cancer/data.py


In [3]:
%%writefile breast_cancer/features.py
def get_features_and_target(df):
    X = df.drop(columns=["diagnosis", "id"])
    y = df["diagnosis"]
    return X, y

Writing breast_cancer/features.py


In [4]:
%%writefile breast_cancer/validate.py
def validate_measurements(measurements):
    ranges = {
        "radius_mean": (5, 40),
        "texture_mean": (0, 60),
        "perimeter_mean": (30, 300),
        "area_mean": (100, 3000),
        "smoothness_mean": (0, 1),
    }

    for name, (minimum, maximum) in ranges.items():
        value = measurements[name]

        if not minimum <= value <= maximum:
            raise ValueError(
                f"{name} must be between {minimum} and {maximum}"
            )

    return True

Writing breast_cancer/validate.py


In [5]:
%%writefile breast_cancer/model.py
import joblib
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


def train_and_save_model(X, y, output_path="breast_cancer_model.joblib"):
    models = {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(max_iter=2000))
        ]),
        "Decision Tree": DecisionTreeClassifier(
            max_depth=5,
            random_state=42
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=200,
            random_state=42
        )
    }

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    results = []

    for name, model in models.items():
        scores = cross_val_score(
            model,
            X,
            y,
            cv=cv,
            scoring="accuracy"
        )

        results.append({
            "Model": name,
            "CV Accuracy": scores.mean()
        })

    results_df = pd.DataFrame(results)
    best_name = results_df.loc[
        results_df["CV Accuracy"].idxmax(),
        "Model"
    ]

    best_model = models[best_name]
    best_model.fit(X, y)

    bundle = {
        "model": best_model,
        "feature_names": list(X.columns),
        "model_name": best_name,
        "cv_results": results_df
    }

    joblib.dump(bundle, output_path)

    print(results_df)
    print(f"\nBest model: {best_name}")
    print(f"Saved to: {output_path}")

    return bundle

Writing breast_cancer/model.py


In [6]:
%%writefile breast_cancer/metrics.py
import joblib
import pandas as pd

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


def evaluate_saved_model(
    data_path="data/breast-cancer.csv",
    model_path="breast_cancer_model.joblib"
):
    df = pd.read_csv(data_path)

    X = df.drop(columns=["diagnosis", "id"])
    y = df["diagnosis"]

    bundle = joblib.load(model_path)
    model = bundle["model"]

    _, X_test, _, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    predictions = model.predict(X_test)

    print("Model:", bundle["model_name"])
    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, predictions))

    print("\nClassification report:")
    print(classification_report(y_test, predictions))

Writing breast_cancer/metrics.py


In [7]:
%%writefile predict.py
import sys
import joblib
import pandas as pd

from breast_cancer.validate import validate_measurements


model_bundle = joblib.load("breast_cancer_model.joblib")
model = model_bundle["model"]
feature_names = model_bundle["feature_names"]

values = [float(value) for value in sys.argv[1:]]

if len(values) != len(feature_names):
    raise ValueError(
        f"Expected {len(feature_names)} measurements, "
        f"but received {len(values)}"
    )

measurements = dict(zip(feature_names, values))
validate_measurements(measurements)

input_data = pd.DataFrame([measurements], columns=feature_names)

prediction = model.predict(input_data)[0]
probabilities = model.predict_proba(input_data)[0]
confidence = probabilities.max() * 100

label = {
    "M": "malignant",
    "B": "benign"
}[prediction]

print("Prediction:", label)
print(f"Confidence: {confidence:.2f}%")

Overwriting predict.py


In [8]:
%%writefile breast_cancer/__init__.py
from .data import load_data
from .features import get_features_and_target
from .model import train_and_save_model
from .validate import validate_measurements
from .metrics import evaluate_saved_model

Writing breast_cancer/__init__.py


In [10]:
from breast_cancer import load_data, get_features_and_target
from breast_cancer.model import train_and_save_model

df = load_data()
X, y = get_features_and_target(df)

bundle = train_and_save_model(X, y)

                 Model  CV Accuracy
0  Logistic Regression     0.973669
1        Decision Tree     0.927993
2        Random Forest     0.954324

Best model: Logistic Regression
Saved to: breast_cancer_model.joblib


In [11]:
from breast_cancer.metrics import evaluate_saved_model

evaluate_saved_model()

Model: Logistic Regression

Confusion matrix:
[[72  0]
 [ 1 41]]

Classification report:
              precision    recall  f1-score   support

           B       0.99      1.00      0.99        72
           M       1.00      0.98      0.99        42

    accuracy                           0.99       114
   macro avg       0.99      0.99      0.99       114
weighted avg       0.99      0.99      0.99       114



In [12]:
!python predict.py 17.99 10.38 122.8 1001 0.1184 0.2776 0.3001 0.1471 0.2419 0.07871 1.095 0.9053 8.589 153.4 0.006399 0.04904 0.05373 0.01587 0.03003 0.006193 25.38 17.33 184.6 2019 0.1622 0.6656 0.7119 0.2654 0.4601 0.1189

Prediction: malignant
Confidence: 100.00%
